In [1]:
import pandas as pd

In [13]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [7]:
url = "https://raw.githubusercontent.com/Jose-guerra2002/Jose_Rigoberto_Guerra_Rodriguez_2510832022/refs/heads/data/readmi/data/clave_B_asociacion.csv"
df = pd.read_csv(url, sep=',')
print(df.columns.tolist())

['transaccion_id', 'cliente_id', 'fecha', 'categoria', 'item', 'cantidad', 'canal']


In [8]:
display(df.head())

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,B-T0001,B-C0076,2026-01-03,Salud,Alcohol,1,Tienda
1,B-T0001,B-C0076,2026-01-03,Medicamentos,Antigripal,1,Tienda
2,B-T0001,B-C0076,2026-01-03,Salud,Mascarilla,1,Tienda
3,B-T0001,B-C0076,2026-01-03,Bebes,Panal,1,Tienda
4,B-T0002,B-C0062,2026-03-26,Salud,Alcohol,1,App


In [17]:
# ver valores nulos por columnas
print("\nvalor nulo por colmna")
print(df.isnull().sum())

#  ver valores duploicados
print("\nvalores duplicados por columna")
print(df.duplicated().sum())

# ver info
print("\nindformacion de el DataFrame")
df.info()


valor nulo por colmna
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

valores duplicados por columna
1

indformacion de el DataFrame
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 603 entries, 0 to 602
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  603 non-null    object
 1   cliente_id      603 non-null    object
 2   fecha           603 non-null    object
 3   categoria       603 non-null    object
 4   item            603 non-null    object
 5   cantidad        603 non-null    int64 
 6   canal           602 non-null    object
dtypes: int64(1), object(6)
memory usage: 33.1+ KB


In [18]:
from mlxtend.preprocessing import TransactionEncoder

# Agrupar los items
transactions = df.groupby('transaccion_id')['item'].apply(list).values

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# Crear un DataFrame
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("Dimensiones del DataFrame codificado:", df_encoded.shape)
print("Primeras 5 filas del DataFrame codificado:")
display(df_encoded.head())

Dimensiones del DataFrame codificado: (170, 20)
Primeras 5 filas del DataFrame codificado:


,Alcohol,Analgesico,Antiacido,Antigripal,Biberon,Bloqueador,Crema,Curitas,Electrolitos,Formula,Gel,Jabon,Mascarilla,Panal,Proteina,Shampoo,Te_relajante,Termometro,Toallitas,Vitaminas
0,True,False,False,True,False,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False
1,True,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,True
3,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,True,False,True,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False


In [19]:
from mlxtend.frequent_patterns import apriori

# agregamos algoritmo Apriori para encontrar los conjuntos de ítems frecuentes y ajustar ek minimo segunsea
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

print("Conjuntos de ítems frecuentes:")
display(frequent_itemsets.head())

Conjuntos de ítems frecuentes:


,support,itemsets
0,0.235294,(Alcohol)
1,0.070588,(Analgesico)
2,0.158824,(Antiacido)
3,0.270588,(Antigripal)
4,0.141176,(Biberon)


In [15]:
from mlxtend.frequent_patterns import association_rules

# Generar las reglas de asociación
# Puedes ajustar las metricas y los umbrales según tus necesidades
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Ordenar las reglas por 'confidence' y 'lift' para ver las más interesantes
rules = rules.sort_values(['confidence', 'lift'], ascending=[False, False])

print("Reglas de asociación generadas:")
display(rules.head(10))

Reglas de asociación generadas (primeras 10 filas):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
36,"(Alcohol, Mascarilla)",(Antigripal),0.141176,0.270588,0.129412,0.916667,3.387681,1.0,0.091211,8.752941,0.820672,0.458333,0.885753,0.697464
37,"(Alcohol, Antigripal)",(Mascarilla),0.152941,0.282353,0.129412,0.846154,2.996795,1.0,0.086228,4.664706,0.786616,0.423077,0.785624,0.652244
38,"(Antigripal, Mascarilla)",(Alcohol),0.170588,0.235294,0.129412,0.758621,3.224138,1.0,0.089273,3.168067,0.831721,0.468085,0.684350,0.654310
31,(Proteina),(Vitaminas),0.247059,0.294118,0.164706,0.666667,2.266667,1.0,0.092042,2.117647,0.742188,0.437500,0.527778,0.613333
0,(Alcohol),(Antigripal),0.235294,0.270588,0.152941,0.650000,2.402174,1.0,0.089273,2.084034,0.763314,0.433333,0.520161,0.607609
9,(Antigripal),(Mascarilla),0.270588,0.282353,0.170588,0.630435,2.232790,1.0,0.094187,1.941869,0.756952,0.446154,0.485032,0.617301
8,(Mascarilla),(Antigripal),0.282353,0.270588,0.170588,0.604167,2.232790,1.0,0.094187,1.842724,0.769361,0.446154,0.457325,0.617301
2,(Alcohol),(Mascarilla),0.235294,0.282353,0.141176,0.600000,2.125000,1.0,0.074740,1.794118,0.692308,0.375000,0.442623,0.550000
1,(Antigripal),(Alcohol),0.270588,0.235294,0.152941,0.565217,2.402174,1.0,0.089273,1.758824,0.800248,0.433333,0.431438,0.607609
30,(Vitaminas),(Proteina),0.294118,0.247059,0.164706,0.560000,2.266667,1.0,0.092042,1.711230,0.791667,0.437500,0.415625,0.613333


In [16]:
display(rules.head(10))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
36,"(Alcohol, Mascarilla)",(Antigripal),0.141176,0.270588,0.129412,0.916667,3.387681,1.0,0.091211,8.752941,0.820672,0.458333,0.885753,0.697464
37,"(Alcohol, Antigripal)",(Mascarilla),0.152941,0.282353,0.129412,0.846154,2.996795,1.0,0.086228,4.664706,0.786616,0.423077,0.785624,0.652244
38,"(Antigripal, Mascarilla)",(Alcohol),0.170588,0.235294,0.129412,0.758621,3.224138,1.0,0.089273,3.168067,0.831721,0.468085,0.684350,0.654310
31,(Proteina),(Vitaminas),0.247059,0.294118,0.164706,0.666667,2.266667,1.0,0.092042,2.117647,0.742188,0.437500,0.527778,0.613333
0,(Alcohol),(Antigripal),0.235294,0.270588,0.152941,0.650000,2.402174,1.0,0.089273,2.084034,0.763314,0.433333,0.520161,0.607609
9,(Antigripal),(Mascarilla),0.270588,0.282353,0.170588,0.630435,2.232790,1.0,0.094187,1.941869,0.756952,0.446154,0.485032,0.617301
8,(Mascarilla),(Antigripal),0.282353,0.270588,0.170588,0.604167,2.232790,1.0,0.094187,1.842724,0.769361,0.446154,0.457325,0.617301
2,(Alcohol),(Mascarilla),0.235294,0.282353,0.141176,0.600000,2.125000,1.0,0.074740,1.794118,0.692308,0.375000,0.442623,0.550000
1,(Antigripal),(Alcohol),0.270588,0.235294,0.152941,0.565217,2.402174,1.0,0.089273,1.758824,0.800248,0.433333,0.431438,0.607609
30,(Vitaminas),(Proteina),0.294118,0.247059,0.164706,0.560000,2.266667,1.0,0.092042,1.711230,0.791667,0.437500,0.415625,0.613333
